# Study 873 — Sentiment Beta 🎭

**Do the stocks that ride euphoria hardest go on to earn *less*?**

Baker & Wurgler (2006, 2007) argue that a stock's **sentiment beta** — how strongly its
returns co-move with market sentiment — flags the speculative, hard-to-value names that
get over-priced in euphoria and **under-perform afterwards**. So a long **low-sentiment-
beta** / short **high-sentiment-beta** book should earn a *positive* spread, widening
**after sentiment peaks**. We take the self-contained daily version on a liquid US
cross-section (2010-01-04 → 2026-06-30, 50 names), proxying sentiment with
a tradable high-minus-low-volatility spread built from the panel itself.

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Survivorship: current-membership mega-caps — magnitudes are an upper
bound.*


## 1. The idea in one picture

A **sentiment gauge** rises in risk-on euphoria (the speculative, high-vol names get bid up) and falls in risk-off. A stock's **sentiment beta** is how hard it rides that mood. The theory: the high-beta euphoria-chasers are over-priced when sentiment is high, so their *future* returns disappoint. Sort on the beta; buy the boring low-beta names, sell the high-beta ones.

In [1]:
import numpy as np, pandas as pd
R = dict(spread_bps=-6.2, t_nw=-2.86, lo_bps=4.82, hi_bps=11.01, gross_sharpe=-0.71,
         gauge_bps=4.74, gauge_annvol=19.4)
print('sentiment gauge (high-minus-low-vol): %+.2f bps/day, %.1f%% ann-vol'
      % (R['gauge_bps'], R['gauge_annvol']))
print('long low-beta / short high-beta spread: %+.2f bps/day (NW t = %+.2f)'
      % (R['spread_bps'], R['t_nw']))
print('  low-beta book %+.2f bps vs high-beta book %+.2f bps'
      % (R['lo_bps'], R['hi_bps']))
print('  gross spread Sharpe (before cost): %.2f' % R['gross_sharpe'])

sentiment gauge (high-minus-low-vol): +4.74 bps/day, 19.4% ann-vol
long low-beta / short high-beta spread: -6.20 bps/day (NW t = -2.86)
  low-beta book +4.82 bps vs high-beta book +11.01 bps
  gross spread Sharpe (before cost): -0.71


## 2. Is the sort just lucky? A live synthetic control

We plant the effect in a seeded toy world — a common sentiment factor with dispersed per-name loadings, and (`edge>0`) high loadings that depress forward returns — and check the detector recovers it, and stays *silent* on the null (`edge=0`, betas present but unpriced). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from sentiment_beta import data, strategy as st
null = st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=873, n_assets=40, n_days=1400))
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0025, seed=873, n_assets=40, n_days=1600))
print('null world   : spread NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: spread NW t = %+.2f  (should light up)' % planted['t_nw'])

null world   : spread NW t = -1.80  (should be ~0)
planted world: spread NW t = +5.86  (should light up)


## 3. The honest verdict — the famous edge does *not* replicate here

On this liquid mega-cap tape the long-low-beta / short-high-beta spread is **-6.20 bps/day** with NW *t* = **-2.86** — significant, but with the **opposite sign** to Baker-Wurgler: here the high-sentiment-beta names (the momentum tech mega-caps that sync with the speculative leg) actually *out-earned* the boring low-beta ones — and by *more* after sentiment peaked (-9.21 vs -4.91 bps). The permutation null centres at 0 with sd 1.32 bps; the observed value is ~4.7σ into the *left* tail. The seeded synthetic control recovers a *planted* Baker-Wurgler relation cleanly, so this is a genuine sign-reversal on the mega-cap survivor universe, not a bug — sentiment beta pays where it is a speculative-small-stock effect, not on 50 mega-caps. **Signal: None** (the claimed edge is absent), **Tradability: Mirage**.